# 03 · Generate Datasets and Compute Statistics

Generate 1M synthetic vectors from each generator and compute the same statistics as notebook 01.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))
from utils import read_fbin, write_fbin
from perturbation import PerturbationGenerator
from gmm_generator import GMMGenerator
from metrics import twonn_lid, hub_skewness, c2st_1nn

N = 1_000_000
Path("../data/synth").mkdir(parents=True, exist_ok=True)
Path("../results").mkdir(parents=True, exist_ok=True)

## Load fitted generators

In [ ]:
gen_perturb = PerturbationGenerator.load("../data/models/perturbation.npz")
gen_gmm = GMMGenerator.load("../data/models/gmm.pkl")
print("Generators loaded.")

## Generate 1M vectors

In [ ]:
print(f"Generating {N:,} vectors from perturbation generator...")
perturb_1m = gen_perturb.generate(N, seed=0)
write_fbin("../data/synth/perturbation_1m.fbin", perturb_1m)
print(f"Generating {N:,} vectors from GMM generator...")
gmm_1m = gen_gmm.generate(N, seed=0)
write_fbin("../data/synth/gmm_1m.fbin", gmm_1m)
print("Done.")

## Statistics comparison

| Metric | LAION (from nb01) | Perturbation | GMM |
|--------|------------------|--------------|-----|
| TwoNN LID | — | — | — |
| N₁₀ skewness | — | — | — |
| C2ST vs LAION | — | — | — |

In [ ]:
sample = read_fbin("../data/laion_sample_100k.fbin")

for name, data in [("Perturbation", perturb_1m[:50000]), ("GMM", gmm_1m[:50000])]:
    lid = twonn_lid(data)
    skew = hub_skewness(data)
    c2 = c2st_1nn(sample, data)
    print(f"{name}: LID={lid:.2f}, hub_skew={skew:.3f}, C2ST={c2:.3f}")

## Norm distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
for ax, (name, data) in zip(axes, [
    ("LAION sample", sample),
    ("Perturbation 1M", perturb_1m[:100000]),
    ("GMM 1M", gmm_1m[:100000]),
]):
    norms = np.linalg.norm(data, axis=1)
    ax.hist(norms, bins=50, edgecolor="none")
    ax.set_title(name)
    ax.set_xlabel("L2 norm")
plt.tight_layout()
plt.savefig("../results/03_norm_comparison.png", dpi=120)
plt.show()